# Prompt Evaluation — Part 2: The Full Loop + a Model Grader

This notebook assembles the complete eval pipeline. Trace the flow: `run_eval` → `run_test_case` → (`run_prompt` to produce output, then `grade_by_model` to score it) → average the scores.

**The new idea here is model-based grading ("LLM-as-judge").** How do you score free-form text output automatically? You ask *another* Claude call to judge it.

- **Use a strong grader prompt.** Give the judge a role ("expert AWS code reviewer"), the original task, and the solution — then ask for a **structured** verdict (strengths, weaknesses, reasoning, score).
- **Order matters: reasoning *before* score.** The judge writes its reasoning first and the number last, so the score is a conclusion drawn from analysis rather than a snap guess. This is chain-of-thought applied to grading.
- **The ` ```json ` prefill + `stop_sequences=["```"]` trick** (in both `chat` calls) forces clean, parseable JSON: we put the opening fence in Claude's mouth and stop generation at the closing fence, so `json.loads` gets exactly the object with no prose around it.
- **Watch the weakness of this approach:** the grader is lenient and subjective (scores cluster around 7–8), and it rewards verbose, essay-style answers. The next notebook adds an objective check to counterbalance this.

In [11]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

**Setup.** `load_dotenv()` pulls your `ANTHROPIC_API_KEY` from a local `.env` file so the key never lives in the notebook. The prompt-under-test and the judge share one `client` and one cheap model (`claude-haiku-4-5`) — sensible for evals, which fire many calls.

In [12]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

**Thin API wrapper.** `add_user_message` / `add_assistant_message` just build the `messages` list; `chat` sends it and returns the text of the first content block. Note the `stop_sequences` parameter — that's the lever that makes the JSON-fence trick below work.

In [13]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

**Dataset generator.** Each case is `{task, format}`. The ` ```json ` assistant prefill *plus* `stop_sequences=["```"]` boxes Claude in: it starts right after the fence and stops at the closing one, so `json.loads` gets exactly the array with no surrounding prose. Remember this trick — the grader reuses it.

In [14]:
# Generate the dataset the FIRST time only — re-running won't clobber it.
# Notebooks 003 and 004 reuse this same file, so scores stay comparable across the series.
import os

if os.path.exists("dataset.json"):
    print("dataset.json already exists — skipping generation. Delete the file to regenerate.")
else:
    dataset = generate_dataset()
    with open("dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)
    print("Generated dataset.json for the first time.")

dataset.json already exists — skipping generation. Delete the file to regenerate.


**Generate once, then freeze.** The existence check writes `dataset.json` only the first time. Freezing the dataset is the whole point: a *fixed* set of cases is what lets you compare scores before and after a prompt change. Delete the file if you deliberately want a fresh set.

In [15]:
# Function to grade a test case + output using a model
import re


def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])

    # LLM-authored JSON is fragile: when the judge echoes the solution's regex or Python,
    # its raw backslashes (\d, \., \w ...) are illegal JSON escapes and json.loads raises
    # "Invalid \escape". If the first parse fails, escape any backslash that isn't a valid
    # JSON escape and try once more.
    try:
        return json.loads(eval_text)
    except json.JSONDecodeError:
        repaired = re.sub(r'\\(?!["\\/bfnrtu])', r"\\\\", eval_text)
        return json.loads(repaired)

**The LLM judge — the heart of this notebook.** How do you auto-score free-form text? Ask another Claude call. The structured output (strengths → weaknesses → reasoning → **score**) forces the model to justify itself *before* committing to a number — chain-of-thought applied to grading. When grading quality really matters, use a stronger model here than the one under test.

In [16]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

**The prompt under test.** This is the thing you're actually evaluating — everything else is scaffolding around it. Right now it's unconstrained free-form, which lets answers ramble into essays and code fences. Keep an eye on that; the next notebook reins it in.

In [17]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

**One case, end-to-end.** Produce the output, then grade it. Keeping "run" and "grade" as separate functions is deliberate — you can swap the prompt or the grader independently without touching the other.

In [18]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

**The scoreboard.** Loops every case and averages the scores into a single number. That number is the signal you watch move up or down as you iterate on the prompt — the entire reason to build an eval.

In [19]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6


**Run it.** Loads the frozen dataset and evaluates end-to-end. Expect scores to cluster around 7–8 — a single lenient judge that rewards verbosity. That bias is exactly what the next two notebooks tighten up.

In [20]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS CloudWatch Log Parser\n\nHere's a solution to parse AWS CloudWatch log entries using regular expressions:\n\n```python\nimport re\nfrom datetime import datetime\nfrom typing import Dict, Optional, Tuple\n\ndef parse_cloudwatch_log(log_entry: str) -> Optional[Dict[str, str]]:\n    \"\"\"\n    Parse an AWS CloudWatch log entry and extract timestamp, log level, and message.\n    \n    Args:\n        log_entry: A CloudWatch log entry string\n        \n    Returns:\n        A dictionary with 'timestamp', 'log_level', and 'message' keys, or None if parsing fails\n    \"\"\"\n    # Pattern to match CloudWatch logs with various formats\n    # Matches: [TIMESTAMP] [LOG_LEVEL] MESSAGE\n    pattern = r'^\\[?(\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}\\.\\d+Z)\\]?\\s+\\[?([A-Z]+)\\]?\\s+(.+)$'\n    \n    match = re.match(pattern, log_entry.strip())\n    \n    if match:\n        return {\n            'timestamp': match.group(1),\n            'log_level': match.group(2),\n 